# task_13 Solution

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

base = Path("../data")


In [ ]:

tables = pd.read_csv(base / "tables.csv")
reservations = pd.read_csv(base / "reservations.csv")
orders = pd.read_csv(base / "orders.csv")

reservations["reservation_time"] = pd.to_datetime(reservations["reservation_time"])
reservations["seated_time"] = pd.to_datetime(reservations["seated_time"])

res = reservations.merge(tables[["table_id", "section", "capacity"]], on="table_id")
q1 = str(res.groupby("section")["no_show"].mean().sort_values(ascending=False).index[0])

walkin = res[(res["channel"] == "walkin") & (res["no_show"] == 0)].copy()
walkin["wait_min"] = (walkin["seated_time"] - walkin["reservation_time"]).dt.total_seconds() / 60
q2 = str(int(walkin["wait_min"].median()))

chk = orders.merge(res[["reservation_id", "reservation_time", "party_size"]], on="reservation_id")
chk["check_per_cover"] = (chk["food_total"] + chk["drink_total"]) / chk["party_size"]
chk["weekday"] = chk["reservation_time"].dt.day_name()
q3 = str(chk.groupby("weekday")["check_per_cover"].mean().sort_values(ascending=False).index[0])

seated = res[res["no_show"] == 0]
q4 = f"{(100 * (seated['party_size'] > seated['capacity']).mean()):.1f}%"

rev = orders.merge(res[["reservation_id", "reservation_time", "party_size", "no_show"]], on="reservation_id")
rev = rev[(rev["no_show"] == 0) & (rev["party_size"] >= 4) & (rev["reservation_time"].dt.dayofweek >= 4) & (rev["reservation_time"].dt.hour >= 19)]
rev["total_rev"] = rev["food_total"] + rev["drink_total"]
q5 = str(rev.groupby("server")["total_rev"].sum().sort_values(ascending=False).index[0])


Q1: Which table section has the highest no-show rate?

In [ ]:
print(q1)


Q2: For walk-in reservations that were seated, what is the median waiting time in minutes from reservation_time to seated_time?

In [ ]:
print(q2)


Q3: Which day of the week has the highest average check per cover, where check per cover = (food_total + drink_total) / party_size?

In [ ]:
print(q3)


Q4: What percentage of seated reservations had party_size greater than the assigned table's capacity, rounded to 1 decimal place?

In [ ]:
print(q4)


Q5: Among non-no-show reservations with party_size >= 4 on Friday through Sunday at 19:00 or later, which server generated the most total revenue?

In [ ]:
print(q5)
